In [1]:
import numpy as np
import pandas as pd
import pickle
import random
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*feature names.*")
from feature_engineering import (
    resolve_team_original_to_updated,
    combine_teams_elo, add_match_features, is_home_advantage,
    get_team_hist
    )
from helpers import run_mc_pipeline, fill_predictions_df


In [2]:
group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures.head(5)

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [3]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots.head(5)

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I


## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [4]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

group_predictions.head(3)

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",None,None,None,None,None,None
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",None,None,None,None,None,None
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",None,None,None,None,None,None


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [5]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

knockout_predictions.head(3)

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,None,None,None,None,None,None,None,None,None
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,None,None,None,None,None,None,None,None,None
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),None,None,None,None,None,None,None,None,None


In [6]:
elo_ratings = pd.read_csv('data/elo.csv', on_bad_lines="skip")

In [7]:
rawdf = pd.read_csv(r"data/raw_history.csv")
rawdf["date"] = pd.to_datetime(rawdf["date"], utc=True)
team_hist = get_team_hist(rawdf)

## **Match Simulation Setup**
- build_lambda_cache_group_stage
- build_lambda_cache_knockout
- simulate_match
- group_simulate
- get_round_of_32
- knockout_simulate
- wc_simulate
- monte_carlo_simulate

In [8]:
from helpers import build_lambda_cache_group_stage, build_lambda_cache_knockout, fill_knockout_table
from simulations import simulate_match, simulate_group_stage, get_round_of_32, knockout_simulate

In [9]:
def world_cup_simulation(
    stage_table_df,
    knockout_df,
    simulate_match_fn,
    models,
    elo_ratings,
    team_hist,
    feature_engine,
    cache,
    verbose=False,
    seed=None
    ):
    """ Run a complete FIFA World Cup 2026 simulation. """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    if verbose:
        banner = "  🏆  FIFA WORLD CUP 2026 — FULL SIMULATION  🏆"
        print(f"\n{'═' * 60}")
        print(banner)
        print(f"  Hosted by USA · CANADA · MEXICO")
        print(f"  48 Teams  |  12 Groups  |  R32 → Final")
        print(f"{'═' * 60}")

    # ── Group Stage ──
    # Run simulate_group_stage
    group_stage_return = simulate_group_stage(
        stage_table_df,
        cache,
        simulate_match_fn,
        verbose
    )
    group_stats = group_stage_return["group_stats"]
    
    if verbose:
        print("##################### DONE WITH GROUP STAGE SIMULATION #####################")

    # Run get_round_of_32
    get_round_of_32_return = get_round_of_32(
        group_stats,
        verbose
    )


    # ==================================
    # ROUND OF 32
    # ==================================
    round_of_32_matches = knockout_df[knockout_df["round"] == "Round of 32"].copy()
    round_of_32_matches = fill_knockout_table(
        round_of_32_matches,
        "Round of 32",
        get_round_of_32_return["r32"],
        None
    )

    # Make sure We only have the matches that are in the round of 32
    round_of_32_matches = round_of_32_matches[round_of_32_matches["round"] == "Round of 32"].copy()

    # Build cache for round of 32
    round_of_32_cache = build_lambda_cache_knockout(round_of_32_matches, models, elo_ratings, team_hist, feature_engine)
    
    # Run knockout_simulate
    r32_return = knockout_simulate(
        "Round of 32",
        round_of_32_matches,
        round_of_32_cache,
        simulate_match_fn,
        verbose
    )
    if verbose:
        print("##################### DONE WITH ROUND OF 32 SIMULATION #####################")
    
    
    # ==================================
    # ROUND OF 16
    # ==================================
    # Build table for round of 16
    round_of_16_matches = knockout_df[knockout_df["round"] == "Round of 16"].copy()
    round_of_16_matches = fill_knockout_table(
        round_of_16_matches,
        "Round of 16",
        get_round_of_32_return, # Qualifiers argument is only applicable for round of 32
        r32_return
    )

    # Make sure We only have the matches that are in the round of 16
    round_of_16_matches = round_of_16_matches[round_of_16_matches["round"] == "Round of 16"].copy()

    # Build cache for round of 16
    round_of_16_cache = build_lambda_cache_knockout(round_of_16_matches, models, elo_ratings, team_hist, feature_engine)
    
    
    # Run knockout_simulate
    r16_return = knockout_simulate(
        "Round of 16",
        round_of_16_matches,
        round_of_16_cache,
        simulate_match_fn,
        verbose
    )
        
    if verbose:
        print("##################### DONE WITH ROUND OF 16 SIMULATION #####################")
    

    
    # ==================================
    # QUARTER-FINAL
    # ==================================
    # Build table for quarter-final
    qf_matches = knockout_df[knockout_df["round"] == "Quarter-final"].copy()
    qf_matches = fill_knockout_table(
        qf_matches,
        "Quarter-final",
        get_round_of_32_return, # Qualifiers argument is only applicable for round of 32
        r16_return
    )

    # Make sure We only have the matches that are in the Quarter-final
    qf_matches = qf_matches[qf_matches["round"] == "Quarter-final"].copy()

    # Build cache for round of 16
    qf_cache = build_lambda_cache_knockout(qf_matches, models, elo_ratings, team_hist, feature_engine)


    # Run knockout_simulate
    qf_return = knockout_simulate(
        "Quarter-final",
        qf_matches,
        qf_cache,
        simulate_match_fn,
        verbose
    )

    if verbose:
        print("##################### DONE WITH QUARTER-FINAL SIMULATION #####################")

    # ==================================
    # SEMI-FINAL
    # ==================================
    # Build table for Semi-final
    sf_matches = knockout_df[knockout_df["round"] == "Semi-final"].copy()
    sf_matches = fill_knockout_table(
        sf_matches,
        "Semi-final",
        get_round_of_32_return, # Qualifiers argument is only applicable for round of 32
        qf_return
    )

    # Make sure We only have the matches that are in the Semi-final
    sf_matches = sf_matches[sf_matches["round"] == "Semi-final"].copy()

    # Build cache for round of 16
    sf_cache = build_lambda_cache_knockout(sf_matches, models, elo_ratings, team_hist, feature_engine)


    # Run knockout_simulate
    sf_return = knockout_simulate(
        "Semi-final",
        sf_matches,
        sf_cache,
        simulate_match_fn,
        verbose
    )

    if verbose:
        print("##################### DONE WITH SEMI-FINAL SIMULATION #####################")

    # ==================================
    # FINAL
    # ==================================
    # Build table for Final
    f_matches = knockout_df[knockout_df["round"] == "Final"].copy()
    f_matches = fill_knockout_table(
        f_matches,
        "Final",
        get_round_of_32_return, # Qualifiers argument is only applicable for round of 32
        sf_return
    )

    # Make sure We only have the matches that are in the Final
    f_matches = f_matches[f_matches["round"] == "Final"].copy()

    # Build cache for round of 16
    f_cache = build_lambda_cache_knockout(f_matches, models, elo_ratings, team_hist, feature_engine)


    # Run knockout_simulate
    f_return = knockout_simulate(
        "Final",
        f_matches,
        f_cache,
        simulate_match_fn,
        verbose
    )

    if verbose:
        print("##################### DONE WITH FINAL SIMULATION #####################")

    final_home_team = f_return["knockout_results"][0]["predicted_home_team"]
    final_away_team = f_return["knockout_results"][0]["predicted_away_team"]
    final_home_score = f_return["knockout_results"][0]["predicted_home_goals"]
    final_away_score = f_return["knockout_results"][0]["predicted_away_goals"]
    champion = f_return["knockout_results"][0]["match_winner"]
    runner_up = f_return["knockout_results"][0]["match_loser"]

    # ==================================
    # THIRD PLACE
    # ==================================
    # Build table for Third-place playoff
    tp_matches = knockout_df[knockout_df["round"] == "Third-place playoff"].copy()
    tp_matches = fill_knockout_table(
        tp_matches,
        "Third-place playoff",
        get_round_of_32_return, # Qualifiers argument is only applicable for round of 32
        sf_return
    )

    # Make sure We only have the matches that are in the Third-place playoff
    tp_matches = tp_matches[tp_matches["round"] == "Third-place playoff"].copy()

    # Build cache for Third-place playoff
    tp_cache = build_lambda_cache_knockout(tp_matches, models, elo_ratings, team_hist, feature_engine)


    # Run knockout_simulate
    third_place_return = knockout_simulate(
        "Third-place playoff",
        tp_matches,
        tp_cache,
        simulate_match_fn,
        verbose
    )

    if verbose:
        print("##################### DONE WITH THIRD-PLACE PLAYOFF SIMULATION #####################")

    third = third_place_return["knockout_results"][0]["match_winner"]

    # FINAL RESULT - Combine R32, R16, QF, SF, Final and 3rd Place
    final_knockout_results = r32_return["knockout_results"] + r16_return["knockout_results"] + qf_return["knockout_results"] + sf_return["knockout_results"] + third_place_return["knockout_results"] + f_return["knockout_results"]


    if verbose:
        print(f"\n{'🏆' * 30}")
        print(f"                      FINAL")
        print(f"{'🏆' * 30}")
        print(f"  {final_home_team:<16} {final_home_score} - {final_away_score} {final_away_team:<16}")
        print(f"\n  🏆  WORLD CHAMPION :  {champion}")
        print(f"  🥈  RUNNER-UP      :  {runner_up}")
        print(f"  🥉  THIRD PLACE    :  {third}")
        print(f"{'🏆' * 30}\n")

    return {
        "champion": champion, 
        "runner_up": runner_up, 
        "third": third, 
        "group_stage_results": group_stage_return["group_results"],
        "knockout_results": final_knockout_results,
        }

In [10]:
# Load models

with open("models/goal_home.pkl", "rb") as f:
    goal_home_best = pickle.load(f)

with open("models/goal_away.pkl", "rb") as f:
    goal_away_best = pickle.load(f)

with open("models/yellow_home.pkl", "rb") as f:
    yellow_home_best = pickle.load(f)

with open("models/yellow_away.pkl", "rb") as f:
    yellow_away_best = pickle.load(f)

with open("models/red_home.pkl", "rb") as f:
    red_home_best = pickle.load(f)

with open("models/red_away.pkl", "rb") as f:
    red_away_best = pickle.load(f)
 
with open("models/corner_home.pkl", "rb") as f:
    corner_home_best = pickle.load(f)

with open("models/corner_away.pkl", "rb") as f:
    corner_away_best = pickle.load(f)


models = {
    "goal_home": goal_home_best,
    "goal_away": goal_away_best,
    "yellow_home": yellow_home_best,
    "yellow_away": yellow_away_best,
    "red_home": red_home_best,
    "red_away": red_away_best,
    "corner_home": corner_home_best,
    "corner_away": corner_away_best
}

In [11]:
# define feature engine
feature_engine = {
       "combine_elo": combine_teams_elo,
       "home_adv": is_home_advantage,
       "match_features": add_match_features
    }

In [12]:
gf = group_fixtures.copy()
gf["home_team"] = gf["home_team"].map(resolve_team_original_to_updated)
gf["away_team"] = gf["away_team"].map(resolve_team_original_to_updated)

elo_ratings = elo_ratings.copy()
elo_ratings["Team"] = elo_ratings["Team"].replace({"United States":"USA"})

In [13]:
gf["date_utc"] = pd.to_datetime(gf["date_utc"], utc=True)
ks = knockout_slots.copy()
ks["date_utc"] = pd.to_datetime(ks["date_utc"], utc=True)

In [14]:
group_cache = build_lambda_cache_group_stage(gf, models, elo_ratings, team_hist, feature_engine)

In [15]:
simulation_result = world_cup_simulation(
    gf,
    ks,
    simulate_match,
    models,
    elo_ratings,
    team_hist,
    feature_engine,
    group_cache,
    verbose=True,
    seed=42
)


════════════════════════════════════════════════════════════
  🏆  FIFA WORLD CUP 2026 — FULL SIMULATION  🏆
  Hosted by USA · CANADA · MEXICO
  48 Teams  |  12 Groups  |  R32 → Final
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  GROUP A
════════════════════════════════════════════════════════════

  #   Team                      Pts  W    D    L    GF   GA   GD   Corners YC   RC  
------------------------------------------------------------
1   Mexico                       3    1    0    0    4    0    4       4    0    1
2   South Korea                  0    0    0    0    0    0    0       0    0    0
3   Czechia                      0    0    0    0    0    0    0       0    0    0
4   South Africa                 0    0    0    1    0    4   -4       5    3    0

Match Results - Group A:
  Mexico          4 - 0 South Africa

════════════════════════════════════════════════════════════
  GROUP A
═════════

In [16]:
def monte_carlo(elo_ratings, original_group_df, original_knockout_df, n=1000, seed=42):

    # ===========================
    # DATAFRAME SET UP
    # ===========================
    gf = original_group_df.copy()
    gf["home_team"] = gf["home_team"].map(resolve_team_original_to_updated)
    gf["away_team"] = gf["away_team"].map(resolve_team_original_to_updated)
    gf["date_utc"] = pd.to_datetime(gf["date_utc"], utc=True)

    ks = original_knockout_df.copy()
    ks["date_utc"] = pd.to_datetime(ks["date_utc"], utc=True)

    elo_ratings = elo_ratings.copy()
    elo_ratings["Team"] = elo_ratings["Team"].replace({"United States": "USA"})

    champions, runners_up, thirds = [], [], []

    group_stage_sims = {}
    knockout_stage_sims = {}

    print("🔢 RUNNING MONTE CARLO SIMULATION 🔥")

    # ===========================
    # HARD SANITIZER (CRITICAL FIX)
    # ===========================
    def sanitize(v):
        if v is None:
            return None
        if isinstance(v, str):
            if "Group" in v or "Best 3rd" in v:
                return None
        return v

    for i in range(n):
        s = seed + i if seed is not None else None

        r = world_cup_simulation(
            gf,
            ks,
            simulate_match,
            models,
            elo_ratings,
            team_hist,
            feature_engine,
            group_cache,
            False,
            s
        )

        # ===========================
        # CLEAN SIMULATION OUTPUT HERE
        # ===========================
        gs_clean = [
            {
                k: sanitize(v)
                for k, v in match.items()
            }
            for match in r["group_stage_results"]
        ]


        ks_clean = [
            {
                k: sanitize(v)
                for k, v in match.items()
            }
            for match in r["knockout_results"]
        ]

        group_stage_sims[i] = gs_clean
        knockout_stage_sims[i] = ks_clean

        # ===========================
        # FINAL OUTCOMES (already clean)
        # ===========================
        champions.append(sanitize(r["champion"]))
        runners_up.append(sanitize(r["runner_up"]))
        thirds.append(sanitize(r["third"]))

    # =========================
    # TEAM FREQUENCIES
    # =========================
    c_cnt = pd.Series(champions).value_counts()
    r_cnt = pd.Series(runners_up).value_counts()
    t_cnt = pd.Series(thirds).value_counts()

    # =========================
    # MC MATCH AGGREGATION
    # =========================
    # Pass `ks` (slot definitions) so the aggregated knockout bracket is
    # re-chained: later-round teams are linked to the aggregated winners of
    # the matches that feed them instead of each match's independent mode.
    final_pred_dict, matchup_counts = run_mc_pipeline(
        group_stage_sims,
        knockout_stage_sims,
        ks,
        models=models,
        elo_ratings=elo_ratings,
        team_hist=team_hist,
        feature_engine=feature_engine,
        mc_seed=seed,
    )

    # =========================
    # FILL DATAFRAMES
    # =========================
    final_group_fixtures, final_knockout_slots = fill_predictions_df(
        original_group_df,
        original_knockout_df,
        final_pred_dict
    )
    
    # =========================
    # TEAM MATCHUP DATAFRAME (for potential future features/analysis)
    # =========================
    rows_team_matchups = []

    for pair, count in matchup_counts.items():
        team_a, team_b = sorted(pair)
        
        matchup_percentage = round((count / n * 100), 2)
        
        rows_team_matchups.append({
            "Team A": team_a,
            "Team B": team_b,
            "Matchup Count": count,
            "Matchup %": matchup_percentage
        })

    # =========================
    # TEAM PERFORMANCE TABLE
    # =========================
    rows = []
    elo_ratings_dict = elo_ratings.set_index("Team")["Elo"].to_dict()

    for team in elo_ratings_dict.keys():
        rows.append({
            "Team": team,
            "Rating": elo_ratings_dict[team],
            "Champion": int(c_cnt.get(team, 0)),
            "Runner-Up": int(r_cnt.get(team, 0)),
            "3rd Place": int(t_cnt.get(team, 0)),
        })

    df = pd.DataFrame(rows)
    df["Podium"] = df["Champion"] + df["Runner-Up"] + df["3rd Place"]
    df["Win %"] = (df["Champion"] / n * 100).round(1)
    df["Pod %"] = (df["Podium"] / n * 100).round(1)

    df = df.sort_values(
        ["Champion", "Runner-Up", "3rd Place"],
        ascending=False
    ).reset_index(drop=True)

    df.index += 1
    
    df_team_matchups = pd.DataFrame(rows_team_matchups)

    return df, df_team_matchups, final_group_fixtures, final_knockout_slots

In [17]:
def display_monte_carlo(df, n):
    """Pretty-print Monte Carlo results."""
    sep = "  " + "─" * 68
    print(f"\n{'═' * 72}")
    print(f"  📊  MONTE CARLO ANALYSIS  —  {n:,} SIMULATIONS")
    print(f"{'═' * 72}")
    print(f"  {'#':<4} {'Team':<16} {'Elo':>4} {'Wins':>5} {'RU':>4}"
          f" {'3rd':>4} {'Pod':>4} {'Win%':>7} {'Pod%':>7}")
    print(sep)

    shown = 0
    for idx, row in df.iterrows():
        if row["Champion"] > 0 or row["Podium"] >= 5:
            print(f"  {idx:<4} {row['Team']:<16} {row['Rating']:>4}"
                  f" {row['Champion']:>5} {row['Runner-Up']:>4}"
                  f" {row['3rd Place']:>4} {row['Podium']:>4}"
                  f" {row['Win %']:>6.1f}% {row['Pod %']:>6.1f}%")
            shown += 1
        if shown >= 25:
            break

    if shown < len(df):
        print(f"\n  ... and {48 - shown} more teams with 0 podium finishes.")

    top = df.iloc[0]
    print(f"\n  🏆  Most Likely Champion:  {top['Team']}"
          f"  ({top['Win %']}% win rate)")
    print(f"{'═' * 72}\n")

## ▶️ Monte Carlo Simulation Kick-in

* ***Note:*** Due to heavy computation, the runtime may be long, depending on the number of iterations

In [18]:
num_of_iter = 10

In [19]:
df, df_team_matchups,final_gf, final_ks = monte_carlo(elo_ratings.copy(), group_fixtures.copy(), knockout_slots.copy(), num_of_iter)

🔢 RUNNING MONTE CARLO SIMULATION 🔥


In [20]:
display_monte_carlo(df, num_of_iter)


════════════════════════════════════════════════════════════════════════
  📊  MONTE CARLO ANALYSIS  —  10 SIMULATIONS
════════════════════════════════════════════════════════════════════════
  #    Team              Elo  Wins   RU  3rd  Pod    Win%    Pod%
  ────────────────────────────────────────────────────────────────────
  1    Spain            2165     3    0    0    3   30.0%   30.0%
  2    England          2020     2    0    0    2   20.0%   20.0%
  3    Brazil           1984     1    1    0    2   10.0%   20.0%
  4    Colombia         1975     1    1    0    2   10.0%   20.0%
  5    Croatia          1930     1    0    1    2   10.0%   20.0%
  6    Germany          1923     1    0    0    1   10.0%   10.0%
  7    Panama           1737     1    0    0    1   10.0%   10.0%

  ... and 41 more teams with 0 podium finishes.

  🏆  Most Likely Champion:  Spain  (30.0% win rate)
════════════════════════════════════════════════════════════════════════



In [21]:
df.to_csv(r"results/monte_carlo_results.csv", index=False)
df_team_matchups.to_csv(r"results/monte_carlo_team_matchups.csv", index=False)

In [22]:
final_gf.to_csv(r"results/final_group_predictions.csv", index=False)
final_gf.head(5)

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",1,0,8,4,1,home
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",2,1,9,4,0,home
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",2,1,9,7,1,home
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",3,1,7,6,0,home
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",2,1,12,3,0,home


In [23]:
final_gf.isna().sum()

match_id                0
group                   0
home_team               0
away_team               0
date_utc                0
venue                   0
predicted_home_goals    0
predicted_away_goals    0
corners                 0
yellow_cards            0
red_cards               0
winning_team            0
dtype: int64

In [24]:
final_ks.to_csv(r"results/final_knockout_predictions.csv", index=False)
final_ks.head(5)

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,South Korea,Switzerland,0,5,5,2,0,away,False
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Brazil,Netherlands,5,2,10,4,0,home,False
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,UEFA Playoff D,2,1,12,3,0,home,False
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Japan,Scotland,1,2,4,4,0,away,False
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Ecuador,Norway,2,1,12,2,0,home,False


In [25]:
final_ks.isna().sum()

match_id                0
round                   0
multiplier              0
date_utc                0
venue                   0
slot_home               0
slot_away               0
predicted_home_team     0
predicted_away_team     0
predicted_home_goals    0
predicted_away_goals    0
corners                 0
yellow_cards            0
red_cards               0
match_winner            0
penalties               0
dtype: int64